# Load dataset

In [1]:
import pandas as pd
df = pd.read_csv('clean_data.csv')

# Split data

## Original data

In [2]:
#Split the original data into features and target
from sklearn.model_selection import train_test_split


X = df.drop('classification',axis=1)
y = df['classification']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,random_state=10, stratify=y)

## Feature selection

### Filter method : correlation-based feature selection

In [119]:
correlation_matrix = df.corr()
correlation = correlation_matrix['classification'].sort_values(ascending=False)
cor_index = list((correlation.index[index]) for index, attr in enumerate(correlation) if attr > 0.5 or attr < -0.5)
X_column_corr = cor_index #Because I want to exclude "classification" which is target.

In [127]:
filter_data = df[X_column_corr]
filter_data

,classification,al,htn,dm,rc,pcv,sg,hemo
0,1,1.0,1,1,5.200000,44.0,1.020,15.4
1,1,4.0,0,0,4.707435,38.0,1.020,11.3
2,1,2.0,0,1,4.707435,31.0,1.010,9.6
3,1,4.0,1,0,3.900000,32.0,1.005,11.2
4,1,2.0,0,0,4.600000,35.0,1.010,11.6
...,...,...,...,...,...,...,...,...
395,0,0.0,0,0,4.900000,47.0,1.020,15.7
396,0,0.0,0,0,6.200000,54.0,1.025,16.5
397,0,0.0,0,0,5.400000,49.0,1.020,15.8
398,0,0.0,0,0,5.900000,51.0,1.025,14.2


### Wrapper method : Stepwise method

In [5]:
from sklearn.ensemble import RandomForestClassifier
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

rf = RandomForestClassifier()

# Initialize the SequentialFeatureSelector
sfs = SFS(rf,
          k_features='best',  # Select the best subset of features
          forward=True,        # Forward selection
          floating=True,
          scoring='accuracy',
          cv=5)

# Fit the SFS on your data
sfs = sfs.fit(X, y)

# Get the selected feature indices
selected_features = list(sfs.k_feature_idx_)
selected_columns = df.columns[list(sfs.k_feature_idx_)+ [len(df.columns) - 1]]

In [107]:
wrapper_data = df[selected_columns]

,sg,al,sod,hemo,rc,htn,classification
0,1.020,1.0,137.528754,15.4,5.200000,1,1
1,1.020,4.0,137.528754,11.3,4.707435,0,1
2,1.010,2.0,137.528754,9.6,4.707435,0,1
3,1.005,4.0,111.000000,11.2,3.900000,1,1
4,1.010,2.0,137.528754,11.6,4.600000,0,1
...,...,...,...,...,...,...,...
395,1.020,0.0,150.000000,15.7,4.900000,0,0
396,1.025,0.0,141.000000,16.5,6.200000,0,0
397,1.020,0.0,137.000000,15.8,5.400000,0,0
398,1.025,0.0,135.000000,14.2,5.900000,0,0


### Embbed method : Rain forest

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf=RandomForestClassifier()

#fit the model
rf.fit(X, y)

# get feature importance scores
importance = rf.feature_importances_

# create a DataFrame to store feature importance scores
feature_importance = pd.DataFrame({'feature': X.columns, 'importance': importance})

# sort the features by importance score in descending order
feature_importance = pd.Series(importance, index=X.columns)

num_features = list(range(2, len(df)))
accuracy = []

for i in num_features:
    top_idx = feature_importance.sort_values(ascending=False)[:i].index
    X_train_i, X_test_i = X_train[top_idx], X_test[top_idx]

    accuracy.append(accuracy_score(y_test, RandomForestClassifier().fit(X_train_i, y_train).predict(X_test_i)))

optimal_num = num_features[accuracy.index(max(accuracy))]
top_idx = feature_importance.sort_values(ascending=False)[:optimal_num].index

In [150]:
embbed_data = df[top_idx.append(pd.Index(['classification']))]

# Train model

In [156]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

models = [
    ('LR', LogisticRegression()),
    ('SVM', SVC(gamma='auto')),
    ('Naive', GaussianNB()),
    ('KNN', KNeighborsClassifier(n_neighbors=2))
]

datasets = {
    'df': df,
    'filter_data': filter_data,
    'wrapper_data': wrapper_data,
    'embbed_data': embbed_data
}

# Open a file in write mode to save the results
with open('classification_reports.txt', 'w') as f:
    for dataset_name, dataset in datasets.items():
        print(f"Current Dataset: {dataset_name}", file=f)

        X = dataset.drop('classification', axis=1)
        y = dataset['classification']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=50)

        for model_name, model in models:
            model.fit(X_train, y_train)

            print(f"Model: {model_name}", file=f)

            # Training data
            y_train_pred = model.predict(X_train)
            train_report = classification_report(y_train, y_train_pred)
            print("Train Classification Report:", file=f)
            print(train_report, file=f)

            # Test data
            y_test_pred = model.predict(X_test)
            test_report = classification_report(y_test, y_test_pred)
            print("Test Classification Report:", file=f)
            print(test_report, file=f)

            print("=" * 40, file=f)

        print("=" * 60, file=f)

C:\Users\Admin\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Admin\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Admin\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in l